# 06 — Monitoring loop and alerts

**Goal.** Close the loop: periodically re-collect (onion + ransomware.live), classify any new docs, diff against what was already in DuckDB, and emit alerts for high-severity hits. Visualize the alert feed with a tiny Streamlit app.

**Inputs.** A populated `db/cti.duckdb` from notebooks 03–05.

**Outputs.**
- A new `alerts` table inside `cti.duckdb`
- `alerts/alerts.jsonl` — append-only feed for downstream consumers
- `06_dashboard.py` — Streamlit script you launch separately

**Scope reminder.** This is a teaching loop. Production monitoring needs job scheduling (cron / systemd timers), proper observability, and on-call routing. Here we just demonstrate the core diff + alert pattern.

In [1]:
import json, time
from pathlib import Path
from datetime import datetime, timezone
import duckdb

DB_PATH = Path("db/cti.duckdb")
ALERT_DIR = Path("alerts")
ALERT_DIR.mkdir(parents=True, exist_ok=True)
ALERT_FILE = ALERT_DIR / "alerts.jsonl"

## 1. Severity policy

We define severity as a function of `(label, score)`. The label space comes from your DarkBERT classifier in cti_301/nb 04 — adjust the mapping to match what your model actually predicts.

In [2]:
HIGH_SEV_LABELS = {
    "Hacking", "Cryptocurrency", "Drugs", "Financial",
    # add the labels your trained model actually uses
}
MIN_SCORE = 0.80

def severity(label, score):
    if label in HIGH_SEV_LABELS and score >= MIN_SCORE:
        return "high"
    if score >= MIN_SCORE:
        return "medium"
    return "low"

## 2. Alerts table

In [3]:
con = duckdb.connect(str(DB_PATH))
con.execute("""
CREATE TABLE IF NOT EXISTS alerts (
    alert_id    TEXT PRIMARY KEY,    -- url_id of the document
    url         TEXT,
    source      TEXT,
    label       TEXT,
    score       DOUBLE,
    severity    TEXT,
    title       TEXT,
    snippet     TEXT,
    fetched_at  TIMESTAMP,
    raised_at   TIMESTAMP
);
""")
print("alerts table ready")

alerts table ready


## 3. The diff: documents that look alert-worthy and aren't already alerted

We pick one row per `dedup_group` so mirrored content doesn't generate duplicate alerts.

In [4]:
candidates = con.execute(f"""
    WITH leaders AS (
        SELECT url_id, url, source, label, score, title, text, fetched_at,
               ROW_NUMBER() OVER (PARTITION BY dedup_group ORDER BY url_id) AS rn
        FROM documents
        WHERE label IS NOT NULL AND score >= {MIN_SCORE}
    )
    SELECT l.url_id, l.url, l.source, l.label, l.score, l.title, l.text, l.fetched_at
    FROM leaders l
    LEFT JOIN alerts a ON a.alert_id = l.url_id
    WHERE l.rn = 1 AND a.alert_id IS NULL
""").fetchall()

print(f"{len(candidates)} candidate alerts")

0 candidate alerts


## 4. Apply the severity policy and persist

In [5]:
now = datetime.now(timezone.utc)
new_alerts = []
for url_id, url, source, label, score, title, text, fetched_at in candidates:
    sev = severity(label, score)
    if sev == "low":
        continue
    new_alerts.append((
        url_id, url, source, label, float(score), sev,
        title or "", (text or "")[:300], fetched_at, now,
    ))

if new_alerts:
    con.executemany("""
        INSERT OR REPLACE INTO alerts
        (alert_id, url, source, label, score, severity, title, snippet, fetched_at, raised_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, new_alerts)

    with ALERT_FILE.open("a") as f:
        for a in new_alerts:
            f.write(json.dumps({
                "alert_id": a[0], "url": a[1], "source": a[2],
                "label": a[3], "score": a[4], "severity": a[5],
                "title": a[6], "snippet": a[7],
                "fetched_at": str(a[8]), "raised_at": str(a[9]),
            }) + "\n")

print(f"raised {len(new_alerts)} new alerts")
print(con.execute("SELECT severity, COUNT(*) FROM alerts GROUP BY severity ORDER BY 2 DESC").fetchdf())

raised 0 new alerts
  severity  count_star()
0   medium             3
1     high             1


## 5. Wrap the loop into one function

`run_cycle()` is what a scheduler (cron, systemd timer, GitHub Action) would call. In this notebook we just call it manually. It assumes notebooks 02 / 04 / 05 are reusable as scripts — for the starter repo we recommend running them as notebooks on your chosen cadence and calling `raise_alerts()` at the end.

In [6]:
def raise_alerts(con):
    """Re-run sections 3-4 and return the count of new alerts."""
    rows = con.execute(f"""
        WITH leaders AS (
            SELECT url_id, url, source, label, score, title, text, fetched_at,
                   ROW_NUMBER() OVER (PARTITION BY dedup_group ORDER BY url_id) AS rn
            FROM documents
            WHERE label IS NOT NULL AND score >= {MIN_SCORE}
        )
        SELECT l.url_id, l.url, l.source, l.label, l.score, l.title, l.text, l.fetched_at
        FROM leaders l
        LEFT JOIN alerts a ON a.alert_id = l.url_id
        WHERE l.rn = 1 AND a.alert_id IS NULL
    """).fetchall()
    raised = 0
    now = datetime.now(timezone.utc)
    for r in rows:
        sev = severity(r[3], r[4])
        if sev == "low":
            continue
        con.execute("""
            INSERT OR REPLACE INTO alerts VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (r[0], r[1], r[2], r[3], float(r[4]), sev,
              r[5] or "", (r[6] or "")[:300], r[7], now))
        raised += 1
    return raised

print("raise_alerts() defined — call it after each re-collect")

raise_alerts() defined — call it after each re-collect


## 6. Streamlit dashboard

Run this from the cti_401 directory:

```bash
streamlit run 06_dashboard.py
```

The next cell writes the script. Don't try to run streamlit from inside the notebook — it expects its own process.

In [7]:
Path("06_dashboard.py").write_text('''\
import duckdb, streamlit as st
from pathlib import Path

DB_PATH = Path("db/cti.duckdb")
st.set_page_config(page_title="CTI 401 monitor", layout="wide")
st.title("CTI 401 — alert feed")

con = duckdb.connect(str(DB_PATH), read_only=True)

c1, c2, c3 = st.columns(3)
c1.metric("docs", con.execute("SELECT COUNT(*) FROM documents").fetchone()[0])
c2.metric("alerts", con.execute("SELECT COUNT(*) FROM alerts").fetchone()[0])
c3.metric("high-sev", con.execute("SELECT COUNT(*) FROM alerts WHERE severity=\'high\'").fetchone()[0])

sev = st.selectbox("severity", ["all", "high", "medium"])
sources = [r[0] for r in con.execute("SELECT DISTINCT source FROM alerts ORDER BY 1").fetchall()]
src = st.selectbox("source", ["all"] + sources)

where = []
if sev != "all": where.append(f"severity = \'{sev}\'")
if src != "all": where.append(f"source = \'{src}\'")
clause = ("WHERE " + " AND ".join(where)) if where else ""

st.dataframe(con.execute(f"""
    SELECT raised_at, severity, source, label, ROUND(score, 3) AS score,
           title, snippet, url
    FROM alerts {clause}
    ORDER BY raised_at DESC
    LIMIT 200
""").fetchdf(), use_container_width=True)
''')
print("wrote 06_dashboard.py — run: streamlit run 06_dashboard.py")

wrote 06_dashboard.py — run: streamlit run 06_dashboard.py


## 7. Where to take this next

This starter ends here on purpose. The natural next steps — none of which fit an 8 GB / 6-notebook scope — are:

- A real scheduler (systemd timer or Airflow) instead of "run the notebooks"
- Asset matching (cti_301/nb 06) plugged in to filter alerts by your customer profile
- Actor / vendor re-identification across messages
- Drift monitoring on the classifier itself

Those are the seeds for cti_501.